# Stateful EasyMagpie codec: chunked acoustic tokens to audio

This notebook loads the standalone codec through its registered vLLM model class and decodes the
EasyMagpie predictor's native `[T, 16]` acoustic-token rows. Each predictor row contains two adjacent
8-codebook codec frames, so it produces **80 ms** of 22.05-kHz audio.

The first row is a one-frame prefill. Later one-row chunks use vLLM decode metadata; chunks with 2–6 rows
use stateful prefill/extend metadata. All calls reuse the same fixed convolution cache pages, so the
waveform chunks concatenate directly without replaying left context or applying a cross-fade.

## 1. Convert the NeMo codec once

From the NeMo repository root:

```bash
examples/tts/easymagpie_codec_vllm/scripts/convert_nemo_to_vllm.sh \
  easymagpietts_NEXT/25fps_spectral_codec_with_bandwidth_extension.nemo \
  /tmp/easymagpie_codec_native
```

The output is a standard local model directory containing `config.json` and `model.safetensors`.
Run this notebook with the `easymagpie-vllm` conda environment as its Jupyter kernel.

In [ ]:
import sys
from pathlib import Path

import torch
from IPython.display import Audio, display


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples/tts/easymagpie_codec_vllm").is_dir():
            return candidate
    raise RuntimeError("Start Jupyter in the NeMo checkout or one of its subdirectories")


REPO_ROOT = find_repo_root()
PLUGIN_ROOT = REPO_ROOT / "examples/tts/easymagpie_codec_vllm"
sys.path.insert(0, str(PLUGIN_ROOT))

CHECKPOINT_DIR = Path("/tmp/easymagpie_codec_native")
TOKEN_FILE = (
    REPO_ROOT
    / "examples/tts/easymagpie_vllm_omni/dumped_codes/concurrency_1/request_0000.pt"
)
assert (CHECKPOINT_DIR / "model.safetensors").is_file(), "Run the conversion command above first"
assert TOKEN_FILE.is_file(), TOKEN_FILE

To generate a fresh token file instead of using the checked-in workspace sample:

```bash
conda run -n easymagpie-vllm python \
  examples/tts/easymagpie_vllm_omni/scripts/benchmark_model.py \
  --model examples/tts/easymagpie_vllm_omni/converted_model_multiturn \
  --audio-codes-dir /tmp/easymagpie_audio_codes \
  --no-warmup -n 1 -c 1 --max-new-tokens 256
```

Then set `TOKEN_FILE` to `/tmp/easymagpie_audio_codes/concurrency_1/request_0000.pt`.
`benchmark_model.py` removes prompt, delay, and EOS rows before saving `audio_codes`.

## 2. Load `[T, 16]` predictor output and verify its ordering

EasyMagpie's `stack_codes` packs each row as
`[c0@2t, c0@2t+1, c1@2t, c1@2t+1, ..., c7@2t, c7@2t+1]`.
The native codec converts this to two time-major `[8]` rows internally.

In [ ]:
from easymagpie_codec_vllm.demo import (
    StatefulCodecRunner,
    load_acoustic_tokens,
    make_chunk_plan,
)
from easymagpie_codec_vllm.packing import unstack_acoustic_codes

tokens = load_acoustic_tokens(TOKEN_FILE)
codec_frames = unstack_acoustic_codes(
    tokens,
    num_codebooks=8,
    frame_stacking_factor=2,
)
print(f"Predictor tokens: {tuple(tokens.shape)}")
print(f"Unstacked codec tokens: {tuple(codec_frames.shape)}")
print("First packed row:", tokens[0].tolist())
print("First two 8-codebook frames:\n", codec_frames[:2])

## 3. Load native weights and decode adaptive chunks

`warmup()` compiles both CUDA paths and clears the state afterward. The measured call times below therefore
reflect steady-state kernels rather than first-use Triton compilation.

In [ ]:
runner = StatefulCodecRunner(CHECKPOINT_DIR, device="cuda")
runner.warmup()

chunk_plan = make_chunk_plan(len(tokens), startup=(1, 1, 2), steady=6)
result = runner.decode(tokens, chunk_plan)

print(f"Chunk plan: {chunk_plan}")
print(f"Waveform: {result.audio.numel():,} samples, {result.audio.numel() / runner.sample_rate:.2f} s")
print()
print("chunk  input frames  audio ms  kernel ms")
for index, (frames, elapsed) in enumerate(zip(result.chunk_sizes, result.elapsed_ms)):
    audio_ms = 1000 * frames * runner.samples_per_frame / runner.sample_rate
    print(f"{index:5d}  {frames:12d}  {audio_ms:8.1f}  {elapsed:9.3f}")

In [ ]:
display(Audio(result.audio.numpy(), rate=runner.sample_rate))

Optional: save exactly what the player receives.

In [ ]:
import wave

output_wav = Path("/tmp/easymagpie_native_chunked.wav")
pcm16 = (result.audio.clamp(-1, 1) * 32767).to(torch.int16).numpy()
with wave.open(str(output_wav), "wb") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)
    wav_file.setframerate(runner.sample_rate)
    wav_file.writeframes(pcm16.tobytes())
print(output_wav)

## 4. Verify that chunking does not change the waveform

This compares `[1, 1, 2, 6, ...]` against one full prefill. Both start from zero state and should match up
to normal floating-point kernel tolerance.

In [ ]:
one_shot = runner.decode(tokens, [len(tokens)])
max_abs_error = (one_shot.audio - result.audio).abs().max().item()
print(f"max |one-shot - chunked| = {max_abs_error:.3e}")
torch.testing.assert_close(one_shot.audio, result.audio, atol=3e-5, rtol=3e-5)